In [1]:
SYSTEM_PROMPT = """你是景点搜索专家。你的任务是根据城市和用户偏好搜索合适的景点。

**重要提示:**
你必须使用工具来搜索景点!不要自己编造景点信息!

**工具调用格式:**
使用maps_text_search工具时,必须严格按照以下格式:
`[TOOL_CALL:amap_maps_text_search:keywords=景点关键词,city=城市名]`

**示例:**
用户: "搜索北京的历史文化景点"
你的回复: [TOOL_CALL:amap_maps_text_search:keywords=历史文化,city=北京]

用户: "搜索上海的公园"
你的回复: [TOOL_CALL:amap_maps_text_search:keywords=公园,city=上海]

**注意:**
1. 必须使用工具,不要直接回答
2. 格式必须完全正确,包括方括号和冒号
3. 参数用逗号分隔"""

In [2]:
import urllib.error
import urllib.request

from langchain.tools import tool

@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL.
    """
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"
    text = raw.decode("utf-8", errors="replace")
    return text

E:\XXXX_CodeTool\anaconda\envs\torch_py314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E:\XXXX_CodeTool\anaconda\envs\torch_py314\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
# 尝试openai兼容的接口。。。✅
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

from pydantic import BaseModel,Field

class ContactInfo(BaseModel):
    """景点信息"""
    keywords: str = Field(description="景点的类别")
    city: str = Field(description="所属城市名")
    isFree: bool = Field(description="是否免费")

llm = ChatOpenAI(
    model="gemma4:latest",
    base_url="http://localhost:11434/v1",   # Ollama 的 OpenAI 兼容端点
    api_key="ollama",                        # 随便填，Ollama 不验证
    temperature=0.7,
)

checkpointer = InMemorySaver()

content = f"""想去五天的广州，帮我规划广州的行程。"""

agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
    response_format=ToolStrategy(ContactInfo),  # ← 唯一需要改的地方
)


agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-lc"}},
)

# print(agent_result["messages"][-1].content_blocks)
# agent_result["structured_response"]

structured = agent_result["structured_response"]
print(structured)
print(f"城市: {structured.city}")
print(f"类别: {structured.keywords}")
print(f"免费: {structured.isFree}")


KeyError: 'structured_response'